# Practical 3, Exercise 2: GAN on CIFAR-10

Training runs on Hábrók via `gan_train.py` (`jobscript.sh`). This notebook collects the results for the report:

| Spec | Section |
|---|---|
| Step 1: Generator (100-d noise → 32×32, 4 transposed convs, output activation matches normalisation) | 1 |
| Step 2: Discriminator (4 strided convs + real/fake head) | 1 |
| Step 3: G and D loss curves over epochs | 2 |
| Step 3: generated images at various epochs | 3 |
| Final samples from the trained generator | 4 |

In [ ]:
import glob
import os
import re

import numpy as np
import matplotlib.pyplot as plt
import matplotlib.image as mpimg
import torch
import torch.nn as nn
from torchvision.utils import make_grid

from gan import Generator, Discriminator

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
NOISE_DIM = 100
os.makedirs('figures', exist_ok=True)

dis_losses = [0.4536877906102003, 0.4687099933246512, 0.4361066649743945, 0.5677884233051113, 0.5590279814953517, 0.6081154337960618, 0.5575374287709935, 0.5201765179862173, 0.5806955145761057, 0.520250548354388, 0.5052484338770488, 0.5112104576154688, 0.44785431396615294, 0.49601240051961953, 0.380368492889161, 0.33433594177292936, 0.32979312157395, 0.34864828713910595, 0.3926628763696573, 0.3165388746494166, 0.421820955881546, 0.26859185860874796, 0.3521044318652142, 0.2991945902924309, 0.14213343202469073, 0.43226037831525865, 0.5400684286221302, 0.36699011217688726, 0.4138384977099382, 0.3477272777769884, 0.1697914941207867, 0.5044627233423018, 0.37143530351497656, 0.34499731860991495, 0.3213038214220878, 0.4270402744503891, 0.29218309947956017, 0.2673276862965859, 0.5332519866012949, 0.32383193027754364, 0.37987373141887865, 0.3385115939619788, 0.28730533364091104, 0.30517360257786375, 0.4181980365431126, 0.18387114992650339, 0.43709797891390234, 0.29859089721095206, 0.19014459941536188, 0.30902795089237695, 0.28969321735393333, 0.2517256262101594, 0.463857428126658, 0.331502949856388, 0.18951890907934582, 0.3182053068101558, 0.47176944563197676, 0.374626634840566, 0.1815184967934477, 0.1764379681696241, 0.3747616327469883, 0.4133867246666935, 0.2971328728419802, 0.21971371305194132, 0.40695932385278866, 0.2840319761391064, 0.3366210087554705, 0.23395441116156931, 0.29654689997491634, 0.27333378796501334, 0.30248299859255096, 0.5324702474056405, 0.27582637618185374, 0.15063049757320965, 0.2883683227334345, 0.38763538038961554, 0.1754325128550052, 0.4517688451602708, 0.31208387080149974, 0.2724844680484907, 0.14144449955332378, 0.3457302010414869, 0.28938455337287894, 0.13965433718142461, 0.6054528385918404, 0.19193616755157045, 0.27340548374091406, 0.4815807606116453, 0.14495724537742458, 0.14997902988902845, 0.602255445484899, 0.14411912151916448, 0.48823896262740385, 0.31123219917549716, 0.15661350674499586, 0.38041238886380224, 0.13870693941372542, 0.36782884518960396, 0.13612882707895216, 0.43431404343970587]
gen_losses = [4.050080984830856, 3.9155668105397905, 3.8996784024092617, 3.0836850803117364, 3.257260698140884, 2.8398488923451124, 2.816507308838927, 2.939355582911141, 2.9690806427780463, 2.8071670948850866, 2.887372858366188, 2.774433932058057, 2.9299908722845878, 3.1178203711918155, 3.2541984857953326, 3.4152419903144544, 3.464859608180669, 3.489770934593921, 3.577008457208166, 3.4008102459566936, 3.3402526838023676, 3.4340472679053033, 3.4836611735577487, 3.534655976736424, 3.781341077113638, 3.461791928767284, 3.261659931631436, 3.0381335737753887, 3.125490500876794, 3.289588526670574, 3.379437956274772, 3.2018934713435425, 3.2596338414111914, 3.4420368050994847, 3.5125808836131984, 3.353204933807672, 3.379246597159274, 3.574905897718759, 2.723430517528738, 3.21202005864102, 3.4750892153503945, 3.3757713751627456, 3.353889543182996, 3.4848972251068573, 3.4253268832892028, 3.406385984347791, 3.421221280850622, 3.3665980501338004, 3.5286117779965305, 3.5006770829156952, 3.4814933164767465, 3.5643014233483346, 3.5199108343548855, 3.350301110318729, 3.546193681809367, 3.4302186772835497, 3.472570338790134, 3.3187398640557704, 3.4920993289169, 3.707740513949978, 3.455853393339381, 3.478502639353142, 3.6094210611238164, 3.4738338011868146, 3.3969358034926107, 3.453643119950988, 3.5294783124320057, 3.4856164978474986, 3.5937425489632453, 3.5937969219319674, 3.5577919916534912, 3.423426181657658, 3.426049623562365, 3.724817609300419, 3.6376119906995066, 3.6030827967488035, 3.6008973608211594, 3.520965300152335, 3.392375031980325, 3.6047226414844697, 3.7812586414570712, 3.6073926161328447, 3.692265775700917, 3.8139001319603043, 3.256832489342315, 3.722173658864839, 3.633589773343838, 3.427047339990279, 3.782079202788217, 3.8568449196766834, 3.368308668887262, 3.7456491978801028, 3.483464807155542, 3.4920117140804625, 3.782195205591163, 3.601686464964735, 3.765142793558082, 3.7505475828538137, 3.820783148006517, 3.571015602899525]

## 1. Architecture checks (Steps 1 and 2)

The data is normalised to [-1, 1] with `Normalize((0.5,0.5,0.5), (0.5,0.5,0.5))`, so the generator ends in `Tanh`.
The discriminator outputs a single logit per image; the sigmoid is applied inside `nn.BCEWithLogitsLoss`.

In [ ]:
gen = Generator(NOISE_DIM).to(device)
dis = Discriminator().to(device)
print(gen)
print(dis)

n_tconv = sum(isinstance(m, nn.ConvTranspose2d) for m in gen.modules())
n_conv = sum(isinstance(m, nn.Conv2d) for m in dis.modules())
print(f'Generator transposed conv layers: {n_tconv}')
print(f'Discriminator conv layers:        {n_conv}')

with torch.no_grad():
    z = torch.randn(8, NOISE_DIM, device=device)
    x = gen(z)
    logits = dis(x)
print(f'G: noise {tuple(z.shape)} -> image {tuple(x.shape)}, range [{x.min():.2f}, {x.max():.2f}]')
print(f'D: image {tuple(x.shape)} -> logit {tuple(logits.shape)}')

n_params = lambda m: sum(p.numel() for p in m.parameters())
print(f'Parameters: G = {n_params(gen):,}, D = {n_params(dis):,}')

## 2. Loss curves (Step 3)

Per-epoch losses are the average over all batches in that epoch, as printed by `gan_train.py` (Hábrók job 32916155) and stored in `dis_losses` / `gen_losses` in the first code cell.
D loss = BCE(real) + BCE(fake); G loss = BCE(D(G(z)), 1) (non-saturating objective).

In [ ]:
d_losses, g_losses = np.array(dis_losses), np.array(gen_losses)
epochs = np.arange(1, len(d_losses) + 1)
print(f'{len(epochs)} epochs')

fig, ax = plt.subplots(1, 2, figsize=(11, 3.8))
ax[0].plot(epochs, d_losses, label='Discriminator')
ax[0].plot(epochs, g_losses, label='Generator')
ax[0].set_xlabel('Epoch'); ax[0].set_ylabel('Average BCE loss')
ax[0].set_title('GAN training losses'); ax[0].legend(); ax[0].grid(alpha=0.3)

# zoomed view of D loss
ax[1].plot(epochs, d_losses, color='C0')
ax[1].set_xlabel('Epoch'); ax[1].set_ylabel('Average BCE loss')
ax[1].set_title('Discriminator loss (zoomed)'); ax[1].grid(alpha=0.3)

plt.tight_layout()
plt.savefig('figures/loss_curves.png', dpi=200, bbox_inches='tight')
plt.show()

## 3. Generated images over training (Step 3)

Snapshots saved by `gan_train.py` during training (16 samples per snapshot).
All snapshots use the same fixed 16 noise vectors, so each position in the grid shows how one sample evolves.

In [ ]:
snapshots = sorted(glob.glob('test_images/image_at_epoch_*.png'))
snap_epochs = [int(re.search(r'(\d+)\.png$', p).group(1)) for p in snapshots]
print(f'Found snapshots at epochs: {snap_epochs}')

cols = 6
rows = int(np.ceil(len(snapshots) / cols))
fig, axes = plt.subplots(rows, cols, figsize=(3 * cols, 3.2 * rows), squeeze=False)
for ax, path, ep in zip(axes.flat, snapshots, snap_epochs):
    ax.imshow(mpimg.imread(path))
    ax.set_title(f'Epoch {ep}')
for ax in axes.flat:
    ax.axis('off')

plt.tight_layout()
plt.savefig('figures/gan_progress.png', dpi=200, bbox_inches='tight')
plt.show()

## 4. Samples from the trained generator

Uses `generator_final.pth`, the generator weights after the last epoch (100).

In [ ]:
ckpt = 'generator_final.pth' if os.path.exists('generator_final.pth') else 'generator_best.pth'
gen.load_state_dict(torch.load(ckpt, map_location=device, weights_only=True))
gen.eval()
print(f'Loaded {ckpt}')

torch.manual_seed(0)
with torch.no_grad():
    samples = gen(torch.randn(64, NOISE_DIM, device=device)).cpu()

grid = make_grid(samples, nrow=8, normalize=True, value_range=(-1, 1))
plt.figure(figsize=(7, 7))
plt.imshow(grid.permute(1, 2, 0))
plt.title(f'64 samples from {ckpt}')
plt.axis('off')
plt.savefig('figures/final_samples.png', dpi=200, bbox_inches='tight')
plt.show()

## 5. Notes for the discussion

- **Losses:** at epoch 1, D = 0.45 and G = 4.05. Over epochs 1–10, D averages 0.53 and G 3.26, a fairly balanced game. By epochs 91–100, D averages 0.32 and G 3.64, so the discriminator slowly gains the upper hand. Over the whole run, D stays within 0.14–0.61 and G within 2.7–4.1. Neither reaches the theoretical equilibrium (D = 2 ln 2 ≈ 1.39, G = ln 2 ≈ 0.69). GAN losses oscillate and don't directly measure image quality.
- **Effect of Adam β₁ = 0.5:** an earlier run with default Adam settings (β₁ = 0.9) had D ≈ 0.02 and G ≈ 8 throughout, so the discriminator dominated. With β₁ = 0.5 (the DCGAN setting) training is much more balanced and the samples are noticeably better.
- **Images:** epoch 1 shows noisy colour blobs with strong checkerboard artifacts (from stride-2 transposed convolutions). By epochs 5–10 there are rough colour layouts and backgrounds. By epoch 100 there is recognisable scene structure (sky/ground) and object-like shapes (animals, vehicles), though still blurry and anatomically wrong. Because the noise is fixed, each sample's evolution can be followed across snapshots.
- **Possible improvements:** DCGAN weight init N(0, 0.02), no BatchNorm on D's first layer, one-sided label smoothing, Upsample + Conv to reduce checkerboard artifacts, a longer schedule, or a lower learning rate for D.